# BR04B — Fit common evidence prompts to PULSE's true multimodal context

Use after BR04A reports `PULSE context budget exceeded` in graph arms. The model loaded correctly; the combined conversation template, visual patches, prompt text and output reserve did not fit. Do **not** increase `runtime['context']`, shorten only graph arms, change the frozen parent config, or mix changed prompts into the original pilot folder.

First run with PULSE in its working environment. This counts the exact LLaVA conversation tokens plus processed image patches, checks every case's prompt, and chooses conservative common evidence caps on validation. It derives a new child run using the **same banks, retrieval results, selected IDs and full evidence cases**. All generators should then use this child config for comparable evidence budgets. The original 3-success/2-error pilot stays untouched.

The native input check still runs for every query image. An image-size change stops before inference. The leakage gate and ordinary journal recovery are retained.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
BASE_CFG = CFG
MODEL_KEY = 'pulse'  # pulse first; later llama_ecg, llama_base, qwen38.
# After PULSE calibration, set this to the printed child config path for every generator.
CONTEXT_CONFIG_PATH = None
# For internal_test/Shaoxing preparation, reuse caps chosen on validation:
FROZEN_VALIDATION_CONFIG = None
if MODEL_KEY != 'pulse' and CONTEXT_CONFIG_PATH is None:
    raise ValueError('Calibrate with PULSE first, then select the same child config for all generators.')


In [ ]:
from generation import model_snapshot, load_generator
MODEL_PATH = None
DOWNLOAD_MODEL = False  # Optional authenticated Hugging Face download into existing cache.
TOKEN_FILE = '/path/to/private/hf_token'  # First line only; never printed.
ENABLE_THINKING = False  # Only Qwen has this switch; primary experiment is False.
LOAD_IN_4BIT = False
PULSE_REPO = None  # Your working PULSE repository when MODEL_KEY='pulse'.
MODEL_PATH = model_snapshot(MODEL_KEY, MODEL_PATH, download=DOWNLOAD_MODEL,
                            token_file=TOKEN_FILE, pin_dir=RUN / 'model_downloads')
runtime = load_generator(CFG, MODEL_KEY, MODEL_PATH, enable_thinking=ENABLE_THINKING,
                         load_in_4bit=LOAD_IN_4BIT, pulse_repo=PULSE_REPO)
print('Loaded:', runtime['model_id'], runtime['revision'])

In [ ]:
from extensions.context_fit import prepare_context_fit
if CONTEXT_CONFIG_PATH is None:
    fit = prepare_context_fit(BASE_CFG, runtime, version='v1', max_new_tokens=384,
                              frozen_context_config=FROZEN_VALIDATION_CONFIG)
    FIT_CFG = fit['config']
    print('Chosen common caps:', fit['caps'])
    print('Visual tokens:', fit['visual_tokens'], '| model context:', fit['model_context'])
    print('Maximum expanded prompt:', fit['maximum_input_tokens'])
    print('CHILD CONFIG:', fit['config_path'])
else:
    FIT_CFG = load_config(CONTEXT_CONFIG_PATH)
    print('Using existing child config:', CONTEXT_CONFIG_PATH)


In [ ]:
from extensions.context_fit import generate_context_fit
from extensions.query_image_archive import generate_archived_context_fit
run_generation = generate_archived_context_fit if FIT_CFG["cohort"] == "shaoxing" else generate_context_fit
from generation import folder_for
MAX_NEW_REPORTS = 5  # Inspect all arms; then 0 for remaining reports.
RETRY_ERRORS = False
MAX_RUN_HOURS = 30.0  # Audit + generation, below allocation time remaining after loading.
MAX_NEW_TOKENS = 1024 if MODEL_KEY == 'qwen38' or (FIT_CFG['cohort'] == 'shaoxing' and MODEL_KEY == 'llama_base') else 384
print(run_generation(FIT_CFG, runtime, max_new=MAX_NEW_REPORTS, retry_errors=RETRY_ERRORS,
    max_run_hours=MAX_RUN_HOURS, max_new_tokens=MAX_NEW_TOKENS))
FOLDER = folder_for(FIT_CFG, MODEL_KEY, ENABLE_THINKING, MAX_NEW_TOKENS)
print('Reports:', FOLDER / 'reports.jsonl')


In [ ]:
from common import latest
from collections import Counter
saved = latest(FOLDER / 'reports.jsonl')
failed = [r for r in saved.values() if r['status'] != 'ok']
print('Failures:', Counter(r.get('error') for r in failed))
for r in list(saved.values())[:5]: print(r['query_id'], r['arm'], r.get('text', r.get('error')))


For other models, restart in that model's working kernel, set `MODEL_KEY` and the printed `CONTEXT_CONFIG_PATH`, then run the loading/generation cells. Do not recalibrate the common caps per generator. Do not use the original larger-context pilot reports in this comparison.

For BR05 evaluation, set its `CONFIG_PATH` to the **child config**, so it evaluates the child reports. Banks, source retrieval and the parent config are unchanged; BR01–BR03 do not need repeating.

For a later cohort: first prepare its source retrieval/packets as usual, set its original `cohort`, then run this notebook with PULSE and `FROZEN_VALIDATION_CONFIG` pointing to the validation child config. This checks the fixed caps rather than tuning them on test/external results. A failure requires investigation, not silently changing frozen caps. Optional k sweeps need a separate budget check for their larger reference pools.